In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import shutil
import math
import os
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, trange
import torchmetrics
import matplotlib.pyplot as plt
from pprint import pprint
import utils as course_utils

# ======== Your controls: change these to run the notebook your way ========
# True: a small run on a tiny generated dataset; grading always uses it.
# False: training on real MNIST (downloaded to ~/ once, about 60 MB).
IS_UNITTEST = True
# Training epochs when IS_UNITTEST = False (about 5 seconds per epoch on a laptop CPU).
EPOCHS = 5
# Device of the full run. 'auto' (the default) takes the first available of
# 'cuda' (NVIDIA GPU), 'xla' (TPU), 'mps' (Apple GPU, only when Python runs on
# macOS itself) and 'cpu'; the cell prints the one it took. The small run
# always uses the CPU.
DEVICE = 'auto'
# ==========================================================================
# Grading always uses the small run, whatever IS_UNITTEST is set to above.
IS_UNITTEST = IS_UNITTEST or course_utils.checker.is_unittest_run()
# Thread count changes the order of float sums; the small run must
# repeat the teacher's numbers on any machine.
if IS_UNITTEST:
    torch.set_num_threads(1)

# -- start --
data_dir = course_utils.checker.get_data_dir()
course_utils.checker.save_npz({'seed': np.array(0)}, data_dir / 'inputs_public.npz')
course_utils.checker.save_npz({'seed': np.array(42)}, data_dir / 'inputs_private.npz')
if not course_utils.checker.is_valid_npz('xinputs.npz'):
    shutil.copyfile(data_dir / 'inputs_public.npz', 'xinputs.npz')
# -- end --

seed = int(course_utils.checker.load_answer('xinputs.npz')['seed'])

import source
# -- start --
# The teacher's notebook runs with the teacher's copy of the shared code.
if course_utils.hidden.detect():
    course_utils.hidden.replace_code()
# -- end --
src = source.code.src
answer = {}

# The device of the training: always the CPU in the small run, DEVICE otherwise.
device = src.utils.device.select_device('cpu' if IS_UNITTEST else DEVICE)
print(f"Training device: {src.utils.device.describe(device)}"
      + (" (the small run always uses the CPU)" if IS_UNITTEST else f" (DEVICE = {DEVICE!r})"))


# Variational Neurons: Networks that Know when They Do not Know

An ordinary network gives one answer, and its softmax is often very sure
even for an input unlike anything it has seen: noise, a rotated digit.

In a **variational** layer, every weight is not a number but a random
variable, $w \sim N(\mu, \sigma^2)$, and the network learns $\mu$ and
$\sigma$ of every weight. Every pass samples new weights, so the same input
gives slightly different outputs. Where the training data pinned the
answer down, the samples agree; where it did not, they disagree, and the
average of their probabilities is unsure. This disagreement is the
*uncertainty* of the network.

**Training.** We maximize the evidence lower bound (ELBO) of variational
inference: the expected log-likelihood of the data minus the
Kullback–Leibler divergence of the weights from a prior $N(0, s^2)$:

$$L = \underbrace{\frac1N \sum_i -\log p(y_i \mid x_i, w)}_{\text{cross-entropy, sampled } w}
+ \frac{1}{N_\text{train}} \sum_{\text{weights}} \mathrm{KL}\big(N(\mu, \sigma^2)\,\|\,N(0, s^2)\big).$$

The KL term pulls every $\sigma$ towards $s$ and every $\mu$ towards 0; the
data term pays for that only where the weight matters. The KL of the whole
network is spread over the $N_\text{train}$ examples of an epoch.

**Files to edit:** [src/models/feedforward/variational.py](../../source/code/src/models/feedforward/variational.py) and
the marked cell of this notebook.


In [ ]:
src.utils.seed.seed_all(0)


# Task 1: The Variational Layer

## Goal

Implement `forward` and `kl` of `VariationalLinear` in
[src/models/feedforward/variational.py](../../source/code/src/models/feedforward/variational.py). The layer has
`mu` and `rho` of shape `[out_features, in_features]`, with
$\sigma = \mathrm{softplus}(\rho) = \log(1 + e^\rho)$ (always positive), and an
ordinary `bias`.

**Forward: the local reparameterization trick.** Sampling a whole weight
matrix for every example is slow. But for a fixed input $x$, the output
$o_j = \sum_k w_{jk} x_k + b_j$ is a sum of independent Gaussians, so it is
itself Gaussian:

$$\mathbb{E}[o_j] = \sum_k \mu_{jk} x_k + b_j, \qquad
\mathrm{Var}[o_j] = \sum_k \sigma_{jk}^2 x_k^2.$$

Sample the output directly: $o = \mathbb{E}[o] + \sqrt{\mathrm{Var}[o]} \cdot \varepsilon$,
$\varepsilon \sim N(0, 1)$ (`torch.randn_like`). `F.linear(x, W)` computes
$x W^T$; the mean is already computed in the file. The gradient flows
through $\mu$ and $\sigma$ to the loss: this is the *reparameterization*.

**KL** of two Gaussians, summed over all weights:

$$\mathrm{KL}\big(N(\mu, \sigma^2)\,\|\,N(0, s^2)\big) = \log\frac{s}{\sigma} + \frac{\sigma^2 + \mu^2}{2 s^2} - \frac12.$$

`VariationalNet` (given) is 784 → 256 → 10 of these layers with a ReLU,
and `kl()` of the network is the sum over its two layers. It moves the
pixels from $[-1, 1]$ to $[0, 1]$ first: the output variance is
$\sum_k \sigma_{jk}^2 x_k^2$, so a black pixel ($x_k = 0$) then adds no
noise, and only the strokes of the digit do.


In [ ]:
# Edit: ../../source/code/src/models/feedforward/variational.py
layer = src.models.feedforward.variational.VariationalLinear(5, 3, prior_std=0.5)
# ^^^ This class should be implemented in this task

with torch.no_grad():
    layer.mu.copy_(torch.linspace(-1, 1, 15).reshape(3, 5))
    layer.rho.copy_(torch.linspace(-3, 0, 15).reshape(3, 5))
check_input = torch.linspace(-2, 2, 10).reshape(2, 5)
torch.manual_seed(seed)
answer['variational_layer'] = {
    'output': layer(check_input),
    'kl': layer.kl(),
}
samples = torch.stack([layer(check_input) for _ in range(2000)])
print('one sample:', answer['variational_layer']['output'])
print('mean of 2000 samples:', samples.mean(dim=0))
print('their std:', samples.std(dim=0))
print('KL:', answer['variational_layer']['kl'])


In [ ]:
course_utils.checker.print_progress(
    'Task 1: The Variational Layer', answer,
    prefixes=('variational_layer',),
    is_small_run=IS_UNITTEST)


# Task 2: The Loss

## Goal

`make_losses(network, n_train)` returns two named losses; `train_model`
minimizes their sum and reports both:

- `ce`: the cross-entropy of `batch['signals']['output']` and
  `batch['data']['label']`;
- `kl`: `network.kl() / n_train`.


In [ ]:
MNIST_train = src.datasets.mnist_simple.MNISTSimpleDataset(train=True, small=IS_UNITTEST, seed=seed)
MNIST_valid = src.datasets.mnist_simple.MNISTSimpleDataset(train=False, small=IS_UNITTEST, seed=seed)
print('training images:', len(MNIST_train), 'validation images:', len(MNIST_valid))

# Edit this cell in task.ipynb.
def make_losses(network, n_train):
    return {'ce': lambda batch: batch['signals']['output'].abs().sum()}

## YOUR CODE HERE
# -- start --
def make_losses(network, n_train):
    return {
        'ce': lambda batch: F.cross_entropy(batch['signals']['output'], batch['data']['label']),
        'kl': lambda batch: network.kl() / n_train,
    }
# -- end --


def accuracy(batch):
    return (batch['postprocessed']['class'] == batch['data']['label']).float().sum(), len(batch['data']['label'])


check_network = src.models.feedforward.variational.VariationalNet()
src.utils.deterministic_init(check_network)
scored_batch = {'signals': {'output': torch.tensor([[2.0, 0.0, -1.0], [0.0, 1.0, 0.5]])},
                'data': {'label': torch.tensor([0, 2])}}
answer['test_loss_val'] = {name: fn(scored_batch) for name, fn in make_losses(check_network, len(MNIST_train)).items()}
pprint(answer['test_loss_val'])


In [ ]:
course_utils.checker.print_progress(
    'Task 2: The Loss', answer,
    prefixes=('test_loss_val',),
    is_small_run=IS_UNITTEST)


# Training

`train_model` with `Adam` (`lr=2e-3`) for `EPOCHS` epochs (one in the small
run). The validation accuracy is measured with one sample of the weights.


In [ ]:
batch_size = 32 if IS_UNITTEST else 128
train_dl = torch.utils.data.DataLoader(
    MNIST_train, batch_size=batch_size, shuffle=True, drop_last=True,
    generator=torch.Generator().manual_seed(0))
valid_dl = torch.utils.data.DataLoader(MNIST_valid, batch_size=1000, shuffle=False)

n_epochs = 1 if IS_UNITTEST else EPOCHS
src.utils.seed.seed_all(0)
network = src.models.feedforward.variational.VariationalNet()
optimizer = torch.optim.Adam(network.parameters(), lr=2.0e-3)
history = src.train_loop.train_model(
    network, n_epochs, train_dl, valid_dl, make_losses(network, len(MNIST_train)), optimizer,
    metrics={'accuracy': accuracy}, run_name='variational',
    device=device)
answer['experiment'] = {name: history['valid_loss'][name] for name in history['valid_loss']}
answer['experiment']['valid_accuracy'] = history['valid_metrics']['accuracy']
print('validation accuracy (one sample) by epoch:', np.round(history['valid_metrics']['accuracy'], 4))
sigmas = torch.cat([module.sigma().flatten() for module in (network.hidden, network.output)])
print(f'sigma of the weights: median {float(sigmas.median()):.3f}, '
      f'10% below {float(sigmas.quantile(0.1)):.3f}, 10% above {float(sigmas.quantile(0.9)):.3f}')


In [ ]:
course_utils.checker.print_progress(
    'Training', answer,
    prefixes=('experiment',),
    is_small_run=IS_UNITTEST)


# Uncertainty

For every input we run the network 20 times and average the softmax
probabilities: the **predictive distribution** $\bar p$. Its entropy
$H = -\sum_k \bar p_k \log \bar p_k$ is 0 when all samples are sure of the
same class and $\log 10 \approx 2.3$ when the answer is a uniform guess.

Three kinds of inputs: validation digits, the same digits rotated by 90°
(still strokes, but no digit the network knows), and uniform noise.


In [ ]:
@torch.no_grad()
def predictive(images, n_samples=20):
    probabilities = torch.stack([network({'data': {'image': images}})['signals']['output'].softmax(dim=1)
                                 for _ in range(n_samples)])
    mean = probabilities.mean(dim=0)
    return mean, probabilities.std(dim=0), -(mean * mean.clamp(min=1e-12).log()).sum(dim=1)


network.eval()
digits = torch.stack([MNIST_valid[index]['image'] for index in range(min(1000, len(MNIST_valid)))])
inputs = {
    'digits': digits,
    'rotated digits': digits.rot90(1, dims=(1, 2)),
    'noise': torch.rand(len(digits), 28, 28, generator=torch.Generator().manual_seed(0)) * 2 - 1,
}
results = {name: predictive(images) for name, images in inputs.items()}

valid_labels = torch.stack([MNIST_valid[index]['label'] for index in range(len(digits))])
print('accuracy of the predictive distribution (20 samples):',
      round(float((results['digits'][0].argmax(dim=1) == valid_labels).float().mean()), 4))

plt.figure(figsize=(8, 3.5))
for name, (_, _, entropy) in results.items():
    plt.hist(entropy, bins=40, range=(0, math.log(10)), alpha=0.6, label=f'{name}: mean {float(entropy.mean()):.2f}')
plt.xlabel('entropy of the predictive distribution')
plt.legend()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(11, 5))
for column, (name, images) in enumerate(inputs.items()):
    mean, std, entropy = results[name]
    axes[0, column].imshow(images[0], cmap='gray')
    axes[0, column].set_title(name)
    axes[0, column].axis('off')
    axes[1, column].bar(range(10), mean[0], yerr=std[0], capsize=2)
    axes[1, column].set_ylim(0, 1)
    axes[1, column].set_xticks(range(10))
    axes[1, column].set_title(f'entropy {float(entropy[0]):.2f}')
plt.tight_layout()
plt.show()


What to look for (full run, 5 epochs, about half a minute):

- the accuracy of one sample peaks at about 97% after 2 epochs and then
  falls to about 94%: the KL term pulls every $\sigma$ towards the prior's
  1, and one sample gets noisier. The average of 20 samples stays at about
  97%; averaging is how a variational network should be used;
- the entropy is low on validation digits (mean about 0.2), higher on
  rotated digits (about 1.0) and highest on noise (about 1.6). The bars
  show why: on a digit, all samples agree on one class; on noise, they
  spread over several, with large error bars;
- the learned $\sigma$ are far from equal: weights the data needs end up
  small and certain, the rest stay close to the prior;
- the uncertainty is not perfect: some rotated digits still get sure
  answers. A network can only doubt along the directions its weights vary.


In [ ]:
course_utils.checker.save_answer(answer)
